In [ ]:
import pandas as pd
from thefuzz import process as fuzzy_process
from tqdm import tqdm
import re

# --- V9 FINAL CONFIGURATION ---
ODOO_NEW_PATH = '/Users/marcel/Desktop/PPP overton comparison/Odoo data NEW.csv'
OVERTON_NEW_PATH = '/Users/marcel/Desktop/PPP overton comparison/Overton data NEW.csv'
FINAL_OUTPUT_PATH_SUMMARY = '/Users/marcel/Desktop/ppp_overton_level_summary.csv'
FINAL_OUTPUT_PATH_DETAILS = '/Users/marcel/Desktop/ppp_overton_level_details.csv'

FUZZY_MATCH_THRESHOLD = 95
PUBLISHED_STATE_VALUE = 'Published'

# List of primary countries for consolidation.
# This ensures we correctly group subdivisions.
EU27_UK_COUNTRIES = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary',
    'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden', 'United Kingdom'
]

def consolidate_overton_country(country_name: str) -> str:
    """
    Consolidates Overton's subdivided country names into a single parent country.
    Example: 'Germany: Bavaria' -> 'Germany'
    """
    if not isinstance(country_name, str):
        return 'Unknown'
    # Special handling for UK, as it's often used as a prefix.
    if country_name.startswith('UK'):
        return 'United Kingdom'
    for parent_country in EU27_UK_COUNTRIES:
        if country_name.startswith(parent_country):
            return parent_country
    # If it's not a subdivision of our target countries, return it as is (e.g., 'EU').
    return country_name

def attempt_reciprocal_match(
    ppp_search_title: str,
    candidate_overton_titles: list,
    candidate_ppp_titles: list,
    threshold: int
) -> dict:
    """
    Performs a high-confidence reciprocal match.
    """
    if not isinstance(ppp_search_title, str) or not ppp_search_title.strip() or not candidate_overton_titles:
        return None

    forward_match = fuzzy_process.extractOne(ppp_search_title, candidate_overton_titles)

    if forward_match and forward_match[1] >= threshold:
        best_overton_title, score = forward_match
        if candidate_ppp_titles:
            reverse_match = fuzzy_process.extractOne(best_overton_title, candidate_ppp_titles)
            if reverse_match and reverse_match[0] == ppp_search_title:
                return {'overton_title': best_overton_title, 'similarity_score': score}
    return None

def main():
    """
    Main analysis script to determine the coverage of different policy levels in Overton.
    """
    print("--- Starting Analysis (V9 - Policy Level Coverage) ---")

    # 1. Load and Prepare Odoo (PPP) Data
    try:
        ppp_df = pd.read_csv(ODOO_NEW_PATH, usecols=['Country', 'Name', 'Name (Native Language)', 'State', 'Level'], engine='python')
        ppp_df.rename(columns={'Name': 'title_en', 'Name (Native Language)': 'title_native'}, inplace=True)
        ppp_df_published = ppp_df[ppp_df['State'] == PUBLISHED_STATE_VALUE].copy()
        # Fill NaN countries with a placeholder for grouping.
        ppp_df_published['Country'].fillna('No Country', inplace=True)
        print(f"Loaded and filtered {len(ppp_df_published)} 'Published' policies from Odoo.")
    except Exception as e:
        print(f"Fatal Error loading Odoo data: {e}"); return

    # 2. Load and Prepare Overton Data
    try:
        overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source country'], engine='python', on_bad_lines='warn')
        overton_df.rename(columns={'Title': 'overton_title', 'Source country': 'overton_country'}, inplace=True)
        overton_df.dropna(subset=['overton_title', 'overton_country'], inplace=True)
        # Apply the consolidation logic to the Overton country data.
        overton_df['consolidated_country'] = overton_df['overton_country'].apply(consolidate_overton_country)
        overton_titles_by_country = overton_df.groupby('consolidated_country')['overton_title'].apply(list).to_dict()
        print(f"Processed {len(overton_df)} policies from Overton and consolidated country names.")
    except Exception as e:
        print(f"Fatal Error loading Overton data: {e}"); return

    # 3. Perform Matching
    results = []
    print("\nBeginning cross-referencing...")
    for _, ppp_row in tqdm(ppp_df_published.iterrows(), total=len(ppp_df_published), desc="Matching Policies"):
        ppp_country = ppp_row['Country']
        match_found = False

        # Define the search spaces for this PPP policy
        search_countries = []
        if ppp_country in EU27_UK_COUNTRIES:
            search_countries.append(ppp_country)
        search_countries.append('EU') # Always search the 'EU' bucket as requested

        # Get all possible PPP titles for reciprocal check
        candidate_ppp_titles = [t for t in [ppp_row['title_en'], ppp_row['title_native']] if pd.notna(t)]

        for search_country in search_countries:
            candidate_overton_titles = overton_titles_by_country.get(search_country, [])
            if not candidate_overton_titles:
                continue

            for ppp_title in candidate_ppp_titles:
                match_info = attempt_reciprocal_match(ppp_title, candidate_overton_titles, candidate_ppp_titles, FUZZY_MATCH_THRESHOLD)
                if match_info:
                    match_found = True
                    break
            if match_found:
                break
        
        results.append({
            'ppp_country': ppp_country,
            'ppp_level': ppp_row['Level'],
            'match_status': 'Match' if match_found else 'No Match'
        })

    results_df = pd.DataFrame(results)

       # 4. Generate and Save Output Tables
    print("\n--- Analysis Complete ---")

    # Table 1: Summary of matches by policy level (unchanged)
    level_summary = results_df.groupby('ppp_level')['match_status'].value_counts().unstack(fill_value=0)
    level_summary['Total'] = level_summary.sum(axis=1)
    level_summary['Match_Percentage'] = (level_summary['Match'] / level_summary['Total'] * 100).round(2)
    
    print("\n--- Table 1: Overton Match Rate by Policy Level ---")
    print(level_summary.to_string())
    level_summary.to_csv(FINAL_OUTPUT_PATH_SUMMARY)
    print(f"Summary table saved to: {FINAL_OUTPUT_PATH_SUMMARY}")

    # Table 2: Updated to be a summary by country only
    country_summary = results_df.groupby('ppp_country')['match_status'].value_counts().unstack(fill_value=0)
    # Ensure 'Match' and 'No Match' columns exist to prevent errors if one is missing
    if 'Match' not in country_summary:
        country_summary['Match'] = 0
    if 'No Match' not in country_summary:
        country_summary['No Match'] = 0
    
    country_summary['Total'] = country_summary['Match'] + country_summary['No Match']
    country_summary['Match_Percentage'] = (country_summary['Match'] / country_summary['Total'] * 100).round(2)

    print("\n--- Table 2: Detailed Breakdown by Country ---")
    # We can drop the 'No Country' category from the final display table for clarity
    print(country_summary.drop('No Country', errors='ignore').to_string())
    country_summary.to_csv(FINAL_OUTPUT_PATH_DETAILS)
    print(f"Detailed breakdown saved to: {FINAL_OUTPUT_PATH_DETAILS}")

if __name__ == '__main__':
    main()
